# Optimization Landscape — Exercises

8 exercises covering Hessian eigenvalue classification, saddle point prevalence, sharpness measurement, mode connectivity, loss landscape visualization, and edge of stability analysis.

| Format            | Description                                  |
| ----------------- | -------------------------------------------- |
| **Problem**       | Markdown cell with task description          |
| **Your Solution** | Code cell with scaffolding                   |
| **Solution**      | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus                |
| ----- | --------- | -------------------- |
| ★     | 1-3       | Core mechanics       |
| ★★    | 4-6       | Deeper theory        |
| ★★★   | 7-8       | AI / ML applications |

### Topic Map

| Topic    | Exercise |
| -------- | -------- |
| Hessian classification | 1 |
| Saddle prevalence | 2 |
| Sharpness measurement | 3 |
| Mode connectivity | 4 |
| Flat vs sharp minima | 5 |
| Loss visualization | 6 |
| Edge of stability | 7 |
| Model merging | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', expected)
        print('  got     :', got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

print('Setup complete.')

---

### Exercise 1: Hessian Eigenvalue Classification ★

Classify critical points of several 2D functions using their Hessian eigenvalues.

**(a)** Compute the Hessian at the origin for each function.

**(b)** Classify each as local min, local max, saddle, or degenerate.

**(c)** Verify that the classification is correct.

In [ ]:
# Exercise 1: Your Solution
functions = {
    'f = x^2 + y^2': np.array([[2, 0], [0, 2]]),
    'f = x^2 - y^2': np.array([[2, 0], [0, -2]]),
    'f = -x^2 - y^2': np.array([[-2, 0], [0, -2]]),
    'f = x^2 - y^2 + xy': np.array([[2, 1], [1, -2]]),
    'f = x^2': np.array([[2, 0], [0, 0]]),
}

for name, H in functions.items():
    eigvals = la.eigvalsh(H)
    # YOUR CODE HERE: classify the critical point
    print(f'{name}: eigenvalues={eigvals}')

In [ ]:
# Exercise 1: Solution
functions = {
    'f = x^2 + y^2': np.array([[2, 0], [0, 2]]),
    'f = x^2 - y^2': np.array([[2, 0], [0, -2]]),
    'f = -x^2 - y^2': np.array([[-2, 0], [0, -2]]),
    'f = x^2 - y^2 + xy': np.array([[2, 1], [1, -2]]),
    'f = x^2': np.array([[2, 0], [0, 0]]),
}

header('Exercise 1: Hessian Eigenvalue Classification')
for name, H in functions.items():
    eigvals = la.eigvalsh(H)
    n_neg = np.sum(eigvals < 0)
    n_zero = np.sum(np.abs(eigvals) < 1e-10)
    n_pos = np.sum(eigvals > 1e-10)
    
    if n_neg == 0 and n_zero == 0:
        ctype = 'Local minimum'
    elif n_neg == 2:
        ctype = 'Local maximum'
    elif n_neg == 1 and n_pos == 1:
        ctype = 'Saddle point'
    elif n_zero > 0:
        ctype = 'Degenerate'
    else:
        ctype = 'Unknown'
    
    print(f'{name}: eigenvalues={eigvals}, type={ctype}')
    
    check_true(f'{name} classified', ctype in ['Local minimum', 'Local maximum', 'Saddle point', 'Degenerate'])

print('\nTakeaway: The Hessian eigenvalue spectrum completely determines the local geometry at a critical point.')

---

### Exercise 2: Saddle Point Prevalence ★

Estimate the probability that a random symmetric matrix has all positive eigenvalues.

**(a)** For $n \in \{2, 3, 5, 10, 20\}$, estimate $P(\text{all } \lambda_i > 0)$ by sampling random symmetric matrices.

**(b)** Compare with the theoretical bound $2^{-n}$.

**(c)** Verify that the probability decays exponentially.

In [ ]:
# Exercise 2: Your Solution
dims = [2, 3, 5, 10, 20]
n_trials = 5000

for n in dims:
    count = 0
    for _ in range(n_trials):
        A = np.random.randn(n, n)
        A = (A + A.T) / 2
        if np.all(la.eigvalsh(A) > 0):
            count += 1
    prob = count / n_trials
    print(f'n={n:>3}: P={prob:.6f}, 2^(-n)={2**(-n):.6f}')

In [ ]:
# Exercise 2: Solution
dims = [2, 3, 5, 10, 20]
n_trials = 5000

header('Exercise 2: Saddle Point Prevalence')
probs = []
for n in dims:
    count = sum(1 for _ in range(n_trials) 
                if np.all(la.eigvalsh((np.random.randn(n,n).T @ np.random.randn(n,n))/n + np.eye(n)) > 0))
    prob = count / n_trials
    probs.append(prob)
    print(f'n={n:>3}: P(all>0)={prob:.6f}, 2^(-n)={2**(-n):.10f}')

check_true('probability decreases with dimension', all(probs[i] >= probs[i+1] for i in range(len(probs)-1)))

print('\nTakeaway: In high dimensions, saddle points overwhelmingly outnumber local minima.')

---

### Exercise 3: Sharpness Measurement ★

Implement the power method to compute the largest Hessian eigenvalue.

**(a)** Implement the power iteration method.

**(b)** Apply it to a simple quadratic function.

**(c)** Verify against the exact eigenvalue.

In [ ]:
# Exercise 3: Your Solution
A = np.diag([1.0, 5.0, 10.0, 50.0])

def hvp(v):
    return A @ v

def power_method(hvp_fn, n, max_iter=100, tol=1e-10):
    v = np.random.randn(n)
    v = v / np.linalg.norm(v)
    for _ in range(max_iter):
        Av = hvp_fn(v)
        lam = v @ Av
        v_new = Av / np.linalg.norm(Av)
        if np.linalg.norm(v_new - v) < tol:
            break
        v = v_new
    return lam, v

lam_est, v_est = power_method(hvp, 4)
lam_exact = la.eigvalsh(A)[-1]
print(f'Estimated lambda_max: {lam_est:.6f}')
print(f'Exact lambda_max: {lam_exact:.6f}')

In [ ]:
# Exercise 3: Solution
A = np.diag([1.0, 5.0, 10.0, 50.0])

def hvp(v): return A @ v

def power_method(hvp_fn, n, max_iter=100, tol=1e-10):
    v = np.random.randn(n)
    v = v / np.linalg.norm(v)
    for _ in range(max_iter):
        Av = hvp_fn(v)
        lam = v @ Av
        v_new = Av / np.linalg.norm(Av)
        if np.linalg.norm(v_new - v) < tol:
            break
        v = v_new
    return lam, v

lam_est, v_est = power_method(hvp, 4)
lam_exact = la.eigvalsh(A)[-1]

header('Exercise 3: Sharpness Measurement')
print(f'Estimated lambda_max: {lam_est:.6f}')
print(f'Exact lambda_max: {lam_exact:.6f}')
check_close('power method converges', lam_est, lam_exact, tol=1e-6)

print('\nTakeaway: The power method efficiently computes the largest eigenvalue without full eigendecomposition.')

---

### Exercise 4: Mode Connectivity ★★

Measure the loss barrier between two independently trained models.

**(a)** Train two quadratic models from different initializations.

**(b)** Measure the loss barrier along the linear interpolation.

**(c)** Verify that the barrier depends on the distance between the models.

In [ ]:
# Exercise 4: Your Solution
np.random.seed(42)
A = np.diag([1.0, 5.0, 10.0])

def f(x): return 0.5 * x @ A @ x

# Train two models
x_a = np.array([1.0, 1.0, 1.0])
for _ in range(100):
    x_a = x_a - 0.01 * (A @ x_a)

np.random.seed(123)
x_b = np.array([-1.0, 1.0, -1.0])
for _ in range(100):
    x_b = x_b - 0.01 * (A @ x_b)

# Linear interpolation
t_vals = np.linspace(0, 1, 50)
path_losses = [f((1-t)*x_a + t*x_b) for t in t_vals]
barrier = max(path_losses) - (f(x_a) + f(x_b)) / 2

print(f'Model A: loss={f(x_a):.6f}')
print(f'Model B: loss={f(x_b):.6f}')
print(f'Barrier: {barrier:.6f}')
print(f'||x_A - x_B||: {np.linalg.norm(x_a - x_b):.4f}')

In [ ]:
# Exercise 4: Solution
np.random.seed(42)
A = np.diag([1.0, 5.0, 10.0])

def f(x): return 0.5 * x @ A @ x

x_a = np.array([1.0, 1.0, 1.0])
for _ in range(100):
    x_a = x_a - 0.01 * (A @ x_a)

np.random.seed(123)
x_b = np.array([-1.0, 1.0, -1.0])
for _ in range(100):
    x_b = x_b - 0.01 * (A @ x_b)

t_vals = np.linspace(0, 1, 50)
path_losses = [f((1-t)*x_a + t*x_b) for t in t_vals]
barrier = max(path_losses) - (f(x_a) + f(x_b)) / 2

header('Exercise 4: Mode Connectivity')
print(f'Model A: loss={f(x_a):.6f}')
print(f'Model B: loss={f(x_b):.6f}')
print(f'Max path loss: {max(path_losses):.6f}')
print(f'Barrier: {barrier:.6f}')
print(f'||x_A - x_B||: {np.linalg.norm(x_a - x_b):.4f}')

check_true('barrier is non-negative', barrier >= 0)
check_true('barrier is finite', barrier < 10)

print('\nTakeaway: The loss barrier measures how disconnected two solutions are. Low barrier enables model merging.')

---

### Exercise 5: Flat vs. Sharp Minima ★★

Compare the sharpness and generalization properties of solutions found by small-batch and large-batch SGD.

**(a)** Train a linear model with batch sizes 1 and 128.

**(b)** Measure the sharpness of each solution.

**(c)** Verify that smaller batches find flatter solutions.

In [ ]:
# Exercise 5: Your Solution
np.random.seed(42)
n, d = 1000, 20
X = np.random.randn(n, d)
y = X @ np.random.randn(d) + 0.1 * np.random.randn(n)

def f_full(w): return 0.5 * np.mean((X @ w - y)**2)
def H_full(): return X.T @ X / n

H = H_full()

# Small batch (B=1)
w_small = np.zeros(d)
for _ in range(10000):
    i = np.random.randint(n)
    w_small = w_small - 0.01 * (X[i] @ w_small - y[i]) * X[i]

# Large batch (B=128)
w_large = np.zeros(d)
for _ in range(10000):
    idx = np.random.randint(0, n, size=128)
    w_large = w_large - 0.01 * X[idx].T @ (X[idx] @ w_large - y[idx]) / 128

# Sharpness via Hessian
sharpness_small = la.eigvalsh(H)[-1]
sharpness_large = la.eigvalsh(H)[-1]

print(f'Small batch: loss={f_full(w_small):.6f}')
print(f'Large batch: loss={f_full(w_large):.6f}')
print(f'Sharpness: {sharpness_small:.6f} (same Hessian, different solutions)')

In [ ]:
# Exercise 5: Solution
np.random.seed(42)
n, d = 1000, 20
X = np.random.randn(n, d)
y = X @ np.random.randn(d) + 0.1 * np.random.randn(n)

def f_full(w): return 0.5 * np.mean((X @ w - y)**2)
H = X.T @ X / n

w_small = np.zeros(d)
for _ in range(10000):
    i = np.random.randint(n)
    w_small = w_small - 0.01 * (X[i] @ w_small - y[i]) * X[i]

w_large = np.zeros(d)
for _ in range(10000):
    idx = np.random.randint(0, n, size=128)
    w_large = w_large - 0.01 * X[idx].T @ (X[idx] @ w_large - y[idx]) / 128

header('Exercise 5: Flat vs Sharp Minima')
print(f'Small batch: loss={f_full(w_small):.6f}, ||w||={np.linalg.norm(w_small):.6f}')
print(f'Large batch: loss={f_full(w_large):.6f}, ||w||={np.linalg.norm(w_large):.6f}')

# Measure effective sharpness via perturbation
rho = 0.1
def perturb_sharpness(w):
    d_dir = np.random.randn(d)
    d_dir = d_dir / np.linalg.norm(d_dir)
    return f_full(w + rho * d_dir) - f_full(w)

sharp_small = np.mean([perturb_sharpness(w_small) for _ in range(100)])
sharp_large = np.mean([perturb_sharpness(w_large) for _ in range(100)])

print(f'\nPerturbation sharpness (rho={rho}):')
print(f'  Small batch: {sharp_small:.6f}')
print(f'  Large batch: {sharp_large:.6f}')

check_true('small batch finds flatter or comparable solution', sharp_small <= sharp_large * 2)

print('\nTakeaway: Smaller batch sizes tend to find flatter minima due to gradient noise.')

---

### Exercise 6: Loss Landscape Visualization ★★

Visualize the 2D loss landscape of a trained model.

**(a)** Train a simple model and save its parameters.

**(b)** Choose two random directions and compute the loss on a 2D grid.

**(c)** Plot the contour map.

In [ ]:
# Exercise 6: Your Solution
np.random.seed(42)
n, d = 200, 10
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = X @ w_true + 0.1 * np.random.randn(n)

def f(w): return 0.5 * np.mean((X @ w - y)**2)

w_star = np.zeros(d)
for _ in range(1000):
    w_star = w_star - 0.01 * X.T @ (X @ w_star - y) / n

# Random directions
d1 = np.random.randn(d); d1 = d1 / np.linalg.norm(d1)
d2 = np.random.randn(d); d2 = d2 - np.dot(d2, d1) * d2; d2 = d2 / np.linalg.norm(d2)

# 2D grid
alpha_vals = np.linspace(-2, 2, 30)
beta_vals = np.linspace(-2, 2, 30)
losses = np.array([[f(w_star + a*d1 + b*d2) for b in beta_vals] for a in alpha_vals])

print(f'Min loss on grid: {losses.min():.6f}')
print(f'Max loss on grid: {losses.max():.6f}')
print(f'Loss at center: {f(w_star):.6f}')

In [ ]:
# Exercise 6: Solution
np.random.seed(42)
n, d = 200, 10
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = X @ w_true + 0.1 * np.random.randn(n)

def f(w): return 0.5 * np.mean((X @ w - y)**2)

w_star = np.zeros(d)
for _ in range(1000):
    w_star = w_star - 0.01 * X.T @ (X @ w_star - y) / n

d1 = np.random.randn(d); d1 = d1 / np.linalg.norm(d1)
d2 = np.random.randn(d); d2 = d2 - np.dot(d2, d1) * d2; d2 = d2 / np.linalg.norm(d2)

alpha_vals = np.linspace(-2, 2, 30)
beta_vals = np.linspace(-2, 2, 30)
losses = np.array([[f(w_star + a*d1 + b*d2) for b in beta_vals] for a in alpha_vals])

header('Exercise 6: Loss Landscape Visualization')
print(f'Min loss on grid: {losses.min():.6f}')
print(f'Max loss on grid: {losses.max():.6f}')
print(f'Loss at center: {f(w_star):.6f}')

check_close('center loss matches minimum', f(w_star), losses.min(), tol=0.1)

if HAS_MPL:
    import matplotlib as mpl
    mpl.rcParams.update({'figure.figsize': (8, 7), 'font.size': 13})
    fig, ax = plt.subplots()
    Alpha, Beta = np.meshgrid(alpha_vals, beta_vals)
    cf = ax.contourf(Alpha, Beta, losses.T, levels=20, cmap='viridis')
    ax.plot(0, 0, 'r*', markersize=15, label='Trained model')
    fig.colorbar(cf, ax=ax)
    ax.set_title('2D loss landscape slice')
    ax.set_xlabel('Direction 1'); ax.set_ylabel('Direction 2')
    ax.legend(); fig.tight_layout(); plt.show()

print('\nTakeaway: Visualization provides intuition for the local geometry of the loss landscape.')

---

### Exercise 7: Edge of Stability ★★★

Track the sharpness during training and verify the edge of stability phenomenon.

**(a)** Train a simple network with GD and track the gradient norm (as a proxy for sharpness).

**(b)** Verify that the sharpness stabilizes near $2/\eta$.

**(c)** Analyze the relationship between learning rate and sharpness.

In [ ]:
# Exercise 7: Your Solution
np.random.seed(42)
n, d = 200, 10
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = X @ w_true + 0.1 * np.random.randn(n)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

H = X.T @ X / n
eta = 0.5
T = 200

w = np.zeros(d)
sharpness_hist = []
loss_hist = []

for t in range(T):
    g = grad(w)
    w = w - eta * g
    
    # Proxy for sharpness: gradient norm change
    g_next = grad(w)
    sharpness = np.linalg.norm(g_next - g) / (eta * np.linalg.norm(g))
    sharpness_hist.append(sharpness)
    loss_hist.append(f(w))

sharpness_hist = np.array(sharpness_hist)

print(f'Learning rate: {eta}')
print(f'Stability threshold: 2/η = {2/eta:.2f}')
print(f'Final sharpness: {sharpness_hist[-1]:.4f}')
print(f'Mean sharpness (last 50): {sharpness_hist[-50:].mean():.4f}')

In [ ]:
# Exercise 7: Solution
np.random.seed(42)
n, d = 200, 10
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = X @ w_true + 0.1 * np.random.randn(n)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

H = X.T @ X / n
eta = 0.5
T = 200

w = np.zeros(d)
sharpness_hist = []
loss_hist = []

for t in range(T):
    g = grad(w)
    w = w - eta * g
    g_next = grad(w)
    sharpness = np.linalg.norm(g_next - g) / (eta * np.linalg.norm(g) + 1e-10)
    sharpness_hist.append(sharpness)
    loss_hist.append(f(w))

sharpness_hist = np.array(sharpness_hist)

header('Exercise 7: Edge of Stability')
print(f'Learning rate: {eta}')
print(f'Stability threshold: 2/η = {2/eta:.2f}')
print(f'Final sharpness: {sharpness_hist[-1]:.4f}')
print(f'Mean sharpness (last 50): {sharpness_hist[-50:].mean():.4f}')

lambda_max = la.eigvalsh(H)[-1]
print(f'True lambda_max: {lambda_max:.4f}')

check_true('sharpness stabilizes', abs(sharpness_hist[-50:].mean() - lambda_max) / lambda_max < 0.5)

if HAS_MPL:
    import matplotlib as mpl
    mpl.rcParams.update({'figure.figsize': (12, 5), 'font.size': 13})
    fig, axes = plt.subplots(1, 2)
    steps = np.arange(1, T+1)
    axes[0].plot(steps, sharpness_hist, color='#0077BB')
    axes[0].axhline(2/eta, color='#CC3311', linestyle='--', label=f'2/η={2/eta:.1f}')
    axes[0].set_title('Sharpness during training'); axes[0].legend()
    axes[1].plot(steps, loss_hist, color='#0077BB')
    axes[1].set_title('Loss during training')
    fig.tight_layout(); plt.show()

print('\nTakeaway: Sharpness self-regulates near 2/η during training.')

---

### Exercise 8: Model Merging ★★★

Explore model merging via linear interpolation and weight averaging.

**(a)** Train three models from different initializations.

**(b)** Compute the pairwise loss barriers.

**(c)** Compare the performance of the averaged model with individual models.

In [ ]:
# Exercise 8: Your Solution
np.random.seed(42)
n, d = 200, 10
X = np.random.randn(n, d)
y = X @ np.random.randn(d) + 0.1 * np.random.randn(n)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

def train_model(seed):
    np.random.seed(seed)
    w = np.random.randn(d) * 0.1
    for _ in range(500):
        w = w - 0.01 * grad(w)
    return w

models = [train_model(i) for i in range(3)]
losses = [f(w) for w in models]

# Weight average
w_avg = np.mean(models, axis=0)
loss_avg = f(w_avg)

print(f'Model 0: loss={losses[0]:.6f}')
print(f'Model 1: loss={losses[1]:.6f}')
print(f'Model 2: loss={losses[2]:.6f}')
print(f'Average: loss={loss_avg:.6f}')

# Pairwise barriers
for i in range(3):
    for j in range(i+1, 3):
        t_vals = np.linspace(0, 1, 20)
        path = [f((1-t)*models[i] + t*models[j]) for t in t_vals]
        barrier = max(path) - (f(models[i]) + f(models[j])) / 2
        print(f'Barrier ({i}-{j}): {barrier:.6f}')

In [ ]:
# Exercise 8: Solution
np.random.seed(42)
n, d = 200, 10
X = np.random.randn(n, d)
y = X @ np.random.randn(d) + 0.1 * np.random.randn(n)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

def train_model(seed):
    np.random.seed(seed)
    w = np.random.randn(d) * 0.1
    for _ in range(500):
        w = w - 0.01 * grad(w)
    return w

models = [train_model(i) for i in range(3)]
losses = [f(w) for w in models]

w_avg = np.mean(models, axis=0)
loss_avg = f(w_avg)

header('Exercise 8: Model Merging')
print(f'Model 0: loss={losses[0]:.6f}')
print(f'Model 1: loss={losses[1]:.6f}')
print(f'Model 2: loss={losses[2]:.6f}')
print(f'Average: loss={loss_avg:.6f}')

barriers = []
for i in range(3):
    for j in range(i+1, 3):
        path = [f((1-t)*models[i] + t*models[j]) for t in np.linspace(0, 1, 20)]
        barrier = max(path) - (f(models[i]) + f(models[j])) / 2
        barriers.append(barrier)
        print(f'Barrier ({i}-{j}): {barrier:.6f}')

check_true('all barriers non-negative', all(b >= 0 for b in barriers))
check_true('average is competitive', loss_avg <= max(losses) * 1.1)

print('\nTakeaway: Weight averaging works when models are connected by low-loss paths.')

---

## What to Review After Finishing

- [ ] Exercise 1: Hessian eigenvalue classification at critical points
- [ ] Exercise 2: Saddle point prevalence scales as 2^(-n)
- [ ] Exercise 3: Power method for efficient sharpness measurement
- [ ] Exercise 4: Loss barrier between independently trained models
- [ ] Exercise 5: Small batches find flatter minima
- [ ] Exercise 6: 2D loss landscape visualization
- [ ] Exercise 7: Edge of stability — sharpness self-regulates
- [ ] Exercise 8: Weight averaging for model merging

## References

1. Li, H. et al. (2018). "Visualizing the loss landscape of neural nets." NeurIPS.
2. Keskar, N. et al. (2017). "On large-batch training for deep learning." ICLR.
3. Cohen, J. et al. (2021). "Gradient descent occurs at the edge of stability." ICLR.
4. Garipov, T. et al. (2018). "Loss surfaces, mode connectivity, and fast ensembling." NeurIPS.
5. Wortsman, M. et al. (2022). "Model soups: Averaging weights of fine-tuned models." ICML.